# Дообучение многоязычной модели E5 для генерации кандидатов Avito

Этот notebook воспроизводит два этапа fine-tuning dense retriever, используемого в основном решении.

Базовая модель: `intfloat/multilingual-e5-small`.

1. **FT1** — одна positive-пара на каждый уникальный query.
2. **FT2** — продолжение обучения FT1; если у query есть другой positive item, на втором этапе используется он.

Оба этапа используют `MultipleNegativesRankingLoss` и `BatchSamplers.NO_DUPLICATES`.

В исходном запуске каждый этап занимал примерно 28 минут на MPS. Для обычного получения `answer.csv` этот notebook запускать не требуется: основной notebook использует готовый `e5_avito_pilot2_final`.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import torch

from datasets import Dataset
from sentence_transformers import (
    SentenceTransformer,
    SentenceTransformerTrainer,
    SentenceTransformerTrainingArguments,
)
from sentence_transformers.losses import MultipleNegativesRankingLoss
from sentence_transformers.training_args import BatchSamplers

DATA_DIR = Path("dataset")
TRAIN_PATH = DATA_DIR / "train.parquet"
ITEMS_PATH = DATA_DIR / "benchmark_items.parquet"

device = (
    "mps" if torch.backends.mps.is_available()
    else "cuda" if torch.cuda.is_available()
    else "cpu"
)
print("device:", device)


## 1. Формирование валидационной выборки

In [ ]:
train_full = pd.read_parquet(TRAIN_PATH)
items = pd.read_parquet(ITEMS_PATH)

SPLIT_RANDOM_STATE = 25
N_SEEN = 907
N_UNSEEN = 1545

rng = np.random.default_rng(SPLIT_RANDOM_STATE)

benchmark_item_ids = set(items["item_id"].astype(str))
available = train_full[
    train_full["item_id"].astype(str).isin(benchmark_item_ids)
].copy()

query_unique_items = train_full.groupby("search_query")["item_id"].nunique()

seen_candidates = available["search_query"].drop_duplicates()
seen_candidates = seen_candidates[
    seen_candidates.map(query_unique_items) >= 2
].to_numpy()

seen_queries = rng.choice(seen_candidates, size=N_SEEN, replace=False)
seen_query_set = set(seen_queries)

unseen_candidates = (
    available.loc[
        ~available["search_query"].isin(seen_query_set),
        "search_query"
    ]
    .drop_duplicates()
    .to_numpy()
)

unseen_queries = rng.choice(unseen_candidates, size=N_UNSEEN, replace=False)
unseen_query_set = set(unseen_queries)

selected_queries = seen_query_set | unseen_query_set
val_pool = available[
    available["search_query"].isin(selected_queries)
].copy()

validation = (
    val_pool
    .sample(frac=1, random_state=SPLIT_RANDOM_STATE)
    .drop_duplicates("search_query")
    .copy()
)

validation["split"] = np.where(
    validation["search_query"].isin(seen_query_set),
    "seen",
    "unseen",
)

validation = validation.rename(columns={"item_id":"relevant_item_id"})
validation = validation[
    [
        "search_query",
        "search_location_id",
        "search_is_delivery_search",
        "search_infm_params_text",
        "search_category",
        "relevant_item_id",
        "split",
    ]
].reset_index(drop=True)

print(validation["split"].value_counts())
print(validation.shape)


## 2. Подготовка train-выборки с исключением скрытых positive-пар

In [ ]:
unseen_queries = set(
    validation.loc[validation["split"] == "unseen", "search_query"]
)

seen_hidden_pairs = (
    validation.loc[
        validation["split"] == "seen",
        ["search_query", "relevant_item_id"]
    ]
    .rename(columns={"relevant_item_id":"item_id"})
    .drop_duplicates()
)

train_ft = train_full[
    ~train_full["search_query"].isin(unseen_queries)
].copy()

train_ft = train_ft.merge(
    seen_hidden_pairs.assign(_hidden=1),
    on=["search_query","item_id"],
    how="left",
)

train_ft = (
    train_ft[train_ft["_hidden"].isna()]
    .drop(columns="_hidden")
    .reset_index(drop=True)
)

print("train_ft:", train_ft.shape)


## 3. Текстовое представление

In [ ]:
def clean_text(x):
    if pd.isna(x):
        return ""
    return str(x).strip()

def build_query(row):
    q = clean_text(row["search_query"])
    p = clean_text(row["search_infm_params_text"])
    return f"query: {q}. {p}" if p else f"query: {q}"

def build_item(row):
    title = clean_text(row["item_title_raw"])
    params = clean_text(row["item_infm_params_text"])
    return f"passage: {title}. {params}" if params else f"passage: {title}"

train_ft["query_text"] = train_ft.apply(build_query, axis=1)
train_ft["positive_text"] = train_ft.apply(build_item, axis=1)


# Этап 1 — FT1

In [ ]:
train_pilot = (
    train_ft
    .sample(frac=1, random_state=42)
    .drop_duplicates("query_text")
    [["query_text","positive_text"]]
    .reset_index(drop=True)
)

print("Pilot pairs:", len(train_pilot))

In [ ]:
train_dataset = Dataset.from_pandas(
    train_pilot.rename(
        columns={
            "query_text":"anchor",
            "positive_text":"positive",
        }
    ),
    preserve_index=False,
)

model_ft = SentenceTransformer(
    "intfloat/multilingual-e5-small",
    device=device,
)
model_ft.max_seq_length = 128

loss = MultipleNegativesRankingLoss(model_ft)

args = SentenceTransformerTrainingArguments(
    output_dir="e5_avito_pilot",
    num_train_epochs=1,
    per_device_train_batch_size=32,
    learning_rate=2e-5,
    warmup_steps=0.1,
    weight_decay=0.01,
    batch_sampler=BatchSamplers.NO_DUPLICATES,
    logging_steps=50,
    save_strategy="no",
    dataloader_num_workers=0,
    fp16=False,
    bf16=False,
    report_to="none",
)

trainer = SentenceTransformerTrainer(
    model=model_ft,
    args=args,
    train_dataset=train_dataset,
    loss=loss,
)


### Обучение(длительный этап): FT1 (~28 мин на MPS в исходном запуске)

In [ ]:
trainer.train()

In [ ]:
model_ft.save("e5_avito_pilot_final")
print("Saved: e5_avito_pilot_final")

# Этап 2 — FT2

In [ ]:
all_pairs = (
    train_ft[["query_text","positive_text"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

first_positive = train_pilot.rename(
    columns={"positive_text":"first_positive"}
)

pilot2_candidates = all_pairs.merge(
    first_positive,
    on="query_text",
    how="left",
)

pilot2_candidates["is_alternative"] = (
    pilot2_candidates["positive_text"]
    != pilot2_candidates["first_positive"]
)

pilot2_candidates = (
    pilot2_candidates
    .sample(frac=1, random_state=43)
    .sort_values(
        "is_alternative",
        ascending=False,
        kind="stable",
    )
)

train_pilot2 = (
    pilot2_candidates
    .drop_duplicates("query_text")
    [["query_text","positive_text","is_alternative"]]
    .reset_index(drop=True)
)

print("Pairs:", len(train_pilot2))
print("Alternative positives:", train_pilot2["is_alternative"].sum())
print("Alternative share:", f"{train_pilot2['is_alternative'].mean():.2%}")

In [ ]:
train_dataset2 = Dataset.from_pandas(
    train_pilot2[["query_text","positive_text"]].rename(
        columns={
            "query_text":"anchor",
            "positive_text":"positive",
        }
    ),
    preserve_index=False,
)

model_ft2 = SentenceTransformer(
    "e5_avito_pilot_final",
    device=device,
)
model_ft2.max_seq_length = 128

loss2 = MultipleNegativesRankingLoss(model_ft2)

args2 = SentenceTransformerTrainingArguments(
    output_dir="e5_avito_pilot2",
    num_train_epochs=1,
    per_device_train_batch_size=32,
    learning_rate=1e-5,
    warmup_steps=0.05,
    weight_decay=0.01,
    batch_sampler=BatchSamplers.NO_DUPLICATES,
    logging_steps=50,
    save_strategy="no",
    dataloader_num_workers=0,
    fp16=False,
    bf16=False,
    report_to="none",
)

trainer2 = SentenceTransformerTrainer(
    model=model_ft2,
    args=args2,
    train_dataset=train_dataset2,
    loss=loss2,
)


### Обучение(длительный этап): FT2 (~28 мин на MPS в исходном запуске)

In [ ]:
trainer2.train()

In [ ]:
model_ft2.save("e5_avito_pilot2_final")
print("Saved: e5_avito_pilot2_final")

## Результат

Основной pipeline использует `e5_avito_pilot2_final/`.

`e5_avito_pilot_final/` — промежуточный checkpoint первого этапа и нужен только для запуска FT2.
